# 📝🤖 Atividade Avaliativa - Classificação de Textos usando *Word Embeddings* e LSTMs


Esta atividade tem como objetivo praticar o uso de *word embeddings* pré-treinados e redes neurais baseadas em LSTMs para classificação de textos. Será utilizada a base de dados [BBC News Archive](https://www.kaggle.com/datasets/hgultekin/bbcnewsarchive) que possui 2225 notícias organizadas nas classes '*business*', '*entertainment*', '*politics*', '*sport*' e '*tech*'.


**As atividades que devem ser realizadas estão nos tópicos:**
- Atividade 1: Pré-Processamento
- Atividade 2: Aplicação de Padding nas Sequências
- Atividade 3: Definição do Modelo
- Atividade 4: Análise dos Resultados


## Preparação do Ambiente e Imports de Bibliotecas

In [1]:

import os
import re

import torch
import unidecode
import kagglehub
import numpy as np
import pandas as pd
import seaborn as sns
import gensim.downloader
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# Ativa o tqdm nas operações do pandas
tqdm.pandas()
# configura o estilo dos plots
sns.set_style('darkgrid')


## Download do Dataset

Vamos utilizar a biblioteca *kagglehub* para fazer o download do dataset e a biblioteca pandas para ler o arquivo `.csv` em um *data frame* e embaralhá-lo.

In [2]:
ds_path = kagglehub.dataset_download("hgultekin/bbcnewsarchive")
csv_path = os.path.join(ds_path, 'bbc-news-data.csv')
df = pd.read_csv(csv_path, sep= '\t')
df = df.sample(frac=1)
df

,category,filename,title,content
2100,tech,277.txt,Tech helps disabled speed demons,An organisation has been launched to encourag...
747,entertainment,238.txt,Portishead back after eight years,Cult British group Portishead have revealed t...
896,politics,001.txt,Labour plans maternity pay rise,Maternity pay for new mothers is to rise by £...
1079,politics,184.txt,Blair rejects Iraq advice calls,Tony Blair has rejected calls for the publica...
1884,tech,061.txt,Concern over RFID tags,Consumers are very concerned about the use of...
...,...,...,...,...
795,entertainment,286.txt,Goodrem wins top female MTV prize,Pop singer Delta Goodrem has scooped one of t...
2180,tech,357.txt,Have hackers recruited your PC?,More than one million computers on the net ha...
2058,tech,235.txt,Internet boom for gift shopping,Cyberspace is becoming a very popular destina...
1925,tech,102.txt,Blogger grounded by her airline,A US airline attendant is fighting for her jo...


## Atividade 1: Pré-Processamento

Use a biblioteca gensim para fazer o download de um modelo de word embeddings pré-treinado. Fique à vontade para escolher diferentes modelos e avaliar suas performances.

In [3]:
# #### ADICIONE SUA RESPOSTA AQUI ####
glove = gensim.downloader.load('glove-twitter-50')

Usando o modelo de *word embeddings*, mostre quais palavras são mais próximas ao termo "economy".

In [4]:
# #### ADICIONE SUA RESPOSTA AQUI ####
glove.most_similar('economy')

[('economic', 0.8814243674278259),
 ('growth', 0.8684114813804626),
 ('government', 0.8332242369651794),
 ('housing', 0.8252440094947815),
 ('climate', 0.8131377696990967),
 ('debt', 0.8097053170204163),
 ('u.s.', 0.8094531297683716),
 ('unemployment', 0.8064248561859131),
 ('policy', 0.8047350645065308),
 ('consumer', 0.8019512891769409)]

Use o modelo de *word embeddings* para recuperar o embedding associado à palavra "championship".

In [8]:
# #### ADICIONE SUA RESPOSTA AQUI ####
glove.get_vector('championship')

array([ 0.63824  ,  0.0032009,  1.3212   , -0.83268  ,  0.98743  ,
        0.49015  ,  0.97442  ,  1.0508   ,  0.39588  , -0.2689   ,
        0.49024  ,  0.27277  , -2.8832   ,  0.80975  ,  1.1724   ,
        0.0079772,  0.10551  ,  0.28507  ,  0.020685 ,  0.16869  ,
        0.65796  ,  0.20325  ,  1.7322   , -0.72003  , -0.17577  ,
        1.0044   ,  0.4387   , -0.84085  , -0.41587  , -0.11564  ,
        0.8604   , -1.0495   ,  0.064151 ,  0.024789 ,  1.3923   ,
       -0.47757  ,  0.23995  , -0.24162  ,  0.11971  ,  0.75676  ,
       -0.29533  , -0.21278  ,  0.99296  ,  0.4212   ,  0.74698  ,
       -0.02645  ,  0.06342  , -0.98884  ,  1.0443   ,  0.26223  ],
      dtype=float32)

Implemente uma função de pré-processamento que realize as operações indicadas nos comentários da função abaixo.


In [10]:
# #### ADICIONE SUA RESPOSTA AQUI ####

MAX_SEQ_LENGTH = 200

def preprocessing(text):
    # Use a biblioteca `unidecode` para substituir caracteres com sinais
    # (acentos, cedilha, etc.) por suas versões sem sinal
    text = unidecode.unidecode(text)

    # Use a biblioteca `re` para substituir caracteres não-alfanuméricos por espaço
    re.sub(r"[^a-zA-Z0-9]", " ", text)

    # Converta o texto para minúsculo
    text = text.lower()

    # Separe o texto em palavras usando o método `split`
    words = text.split()

    # Limite o tamanho da sequencia para o tamanho máximo definido
    # em MAX_SEQ_LENGTH  para reduzir o custo computacional
    words = words[:MAX_SEQ_LENGTH]

    # Converta as palavras em word embeddings usando o modelo da biblioteca gensin criado acima.
    # Lembre-se de verificar se as palavras existem no modelo e ignore aquelas que não existem.
    # Por fim, lembre-se de converter o resultado em um torch.tensor.
    embds = [glove[w] for w in words if w in glove]
    embds = torch.tensor(np.array(embds))

    return embds


# executa a função de pré-processamento para o primeiro texto
print(preprocessing(df['content'].iloc[0]))


tensor([[-0.1130,  0.3805, -0.0350,  ..., -0.3616, -0.1554,  0.1950],
        [ 0.7013,  0.4098, -0.9413,  ...,  0.3887,  0.1079, -1.2859],
        [-0.2947,  0.0772, -0.0316,  ..., -0.4668, -0.6816, -0.7655],
        ...,
        [ 0.1935,  0.4071,  0.1571,  ..., -0.1767, -0.1373, -0.3132],
        [-0.2914,  0.4247,  0.4340,  ..., -0.9268, -0.3602, -0.4726],
        [ 0.2715,  0.3207,  0.1656,  ..., -0.0056, -0.2971,  0.1083]])


A célula abaixo aplica a função de pré-processamento para toda a base de dados. 

In [11]:
x = [preprocessing(t) for t in df['content']]
x

[tensor([[-0.1130,  0.3805, -0.0350,  ..., -0.3616, -0.1554,  0.1950],
         [ 0.7013,  0.4098, -0.9413,  ...,  0.3887,  0.1079, -1.2859],
         [-0.2947,  0.0772, -0.0316,  ..., -0.4668, -0.6816, -0.7655],
         ...,
         [ 0.1935,  0.4071,  0.1571,  ..., -0.1767, -0.1373, -0.3132],
         [-0.2914,  0.4247,  0.4340,  ..., -0.9268, -0.3602, -0.4726],
         [ 0.2715,  0.3207,  0.1656,  ..., -0.0056, -0.2971,  0.1083]]),
 tensor([[ 0.0986,  0.1791, -0.4602,  ..., -0.0285, -0.1044, -0.6391],
         [ 0.3729,  0.4296,  0.7665,  ...,  0.6259, -0.4574, -0.7414],
         [ 0.8828,  1.1352, -0.0503,  ..., -0.4392, -0.1133, -0.4866],
         ...,
         [-0.1037,  0.2117,  0.3113,  ..., -0.2277, -0.1518,  0.1560],
         [ 0.3175,  0.5100,  0.0044,  ...,  0.0315, -0.5050, -0.2420],
         [ 0.2785, -0.1067,  0.7307,  ..., -0.0974,  0.6923, -0.7196]]),
 tensor([[-1.5105, -0.7105,  0.0920,  ..., -0.0053, -0.2375, -0.7601],
         [ 0.0709,  0.5990, -0.4850,  ...,  0

## Atividade 2: Aplicação de Padding nas Sequências

Utilize a operação de *padding* para uniformizar o tamanho das sequências armazenadas na variável `x`. [Leia mais sobre a função pad_sequence](https://docs.pytorch.org/docs/stable/generated/torch.nn.utils.rnn.pad_sequence.html).

In [12]:
from torch.nn.utils.rnn import pad_sequence

x_padded = pad_sequence(x, batch_first=True)

x_padded.shape

torch.Size([2225, 191, 50])

## Divisão dos Dados em Conjuntos de Treino e Teste

Use a classe [LabelEncoder](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.LabelEncoder.html) da biblioteca scikit-learn para transformar os nomes das classes em labels inteiros. Em seguida, transforme a sequência de labels em um tensor.

In [13]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()
y = encoder.fit_transform(df['category'].to_numpy())
y = y.astype(np.int64)
y = torch.tensor(y)
y

tensor([4, 1, 2,  ..., 4, 4, 0])

Use o objeto [LabelEncoder para mostrar a lista de classes identificadas](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.LabelEncoder.html#:~:text=%3E%3E%3E%20list(-,le.classes_,-)%0A%5Bnp.str_).

In [14]:
# #### ADICIONE SUA RESPOSTA AQUI ####
encoder.classes_

array(['business', 'entertainment', 'politics', 'sport', 'tech'],
      dtype=object)

A célula abaixo faz a divisão os dados em conjuntos de treino e teste. Não vamos usar um conjunto de validação para manter a atividade simples.

In [15]:
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(
    x_padded, y,
    test_size=0.2,
    random_state=42,
)

train_ds = TensorDataset(x_train, y_train)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)

test_ds = TensorDataset(x_test, y_test)
test_loader = DataLoader(test_ds, batch_size=64)

print("train size:", len(train_ds), "test size:", len(test_ds))

train size: 1780 test size: 445


## Atividade 3: Definição do Modelo

Complete a definição da classe adicionando os itens especificados nos comentários abaixo.

In [ ]:
# #### ADICIONE SUA RESPOSTA AQUI ####

from torch import nn

# Use o objeto LabelEncoder para recuperar o número de classes
n_classes = ...

class Model(torch.nn.Module):
    def __init__(self):
        super().__init__()

        # Crie uma camada LSTM
        self.lstm = ...

        # Crie uma camada totalmente conectada para classificação.
        # Lembre-se que o tamanho da entrada deve ser igual ao tamanho da LSTM.
        self.fc = ...

    def forward(self, sequence):
        # Passa a sequência de entrada pela LSTM
        ...

        # A saída da LSTM tem o shape [seq_length=1, batch_size, hidden_size].
        # Seq_length=1 porque apenas o estado da última posição da sequência é retornado.
        # A linha abaixo remove a dimensão da sequência fazendo com que o shape se torne [batch_size, hidden_size].
        ...

        # passe a saída da LSTM pela camada totalmente conectada
        prediction = ...

        return prediction

net = Model()
net(x_padded[:3])

## Treinamento e Avaliação

A célula abaixo realiza o treinamento do modelo. Por simplicidade, utilizamos o conjunto de teste para avaliar a generalização do modelo ao longo do treinamento. **Em aplicações reais, o correto seria utilizar um conjunto independente de validação**.

In [ ]:

import torch.optim as optim

net = Model()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(net.parameters(), lr=1e-2)

history = []
n_epochs = 40

# repete por um número de épocas
for epoch in range(n_epochs):
    running_loss = 0.0

    # epoca de treinamento: itera sobre os batches do conjunto de treino
    net.train()
    for i, data in enumerate(train_loader):
        inputs, labels = data

        # Passos para treinamento de um batch
        # #### ADICIONE SUA RESPOSTA AQUI ####
        ...
        loss = ...

        running_loss += loss.item()

    running_loss /= len(train_loader)

    # avaliação no conjunto de teste
    net.eval()
    test_loss = 0.0
    with torch.no_grad():
        for data in test_loader:
            # Compute o erro para um batch de teste
            # #### ADICIONE SUA RESPOSTA AQUI ####
            ...
            test_loss += ...

    test_loss /= len(train_loader)

    history.append([running_loss, test_loss])

    if epoch % max(1, n_epochs // 20)  == 0:
        print(f'[{epoch + 1}] loss: {running_loss:.4f} test loss: {test_loss:.4f}')

print('Finished Training')

A célula abaixo mostra a evolução da função de perda nos conjuntos de treino e teste. É esperado que ambas as curvas reduzam.

In [ ]:
import matplotlib.pyplot as plt

history = np.array(history)

plt.plot(history[:, 0], '-', color='orange', label='train')
plt.plot(history[:, 1], '-', color='blue', label='test')
plt.legend()

Mostre as métricas de performance para o conjunto de teste usando a função classification_report da biblioteca scikit-learn. É possível alcançar uma acurácia superior a `95%`.

In [ ]:

net.eval()

# Itere sobre o conjunto de teste armazenando os labels verdadeiros e as predições
# #### ADICIONE SUA RESPOSTA AQUI ####
true_labels = ...
preds = ...

print(classification_report(true_labels, preds))

A célula abaixo faz uma avaliação qualitativa do modelo do com um conjunto de textos. Verifique se o seu modelo identificou as classes corretamente.

**Importante:** Modifique a linha indicada com o seu objeto do tipo LabelEncoder.

In [ ]:

softmax = torch.nn.Softmax(dim=-1)
net.eval()

def avalia_sentenca(texto):
    embds = preprocessing(texto)

    padded = torch.nn.functional.pad(embds, (0, 0, 0, MAX_SEQ_LENGTH - len(embds)), value=0.0)

    with torch.no_grad():
        padded = torch.unsqueeze(padded, 0)
        logits = net(padded)
        probs = softmax(logits)
        probs = np.squeeze(probs)
        cls = np.argmax(probs)

    print(f"Frase: {texto}")
    print(f"\nProbs: {probs}")

    # Use o objeto do tipo LabelEncoder para recuperar o nome da classe atribuída ao item
    # #### ADICIONE SUA RESPOSTA AQUI ####
    print(f"Class: {...}\n")

    print("------------\n")


# sport
avalia_sentenca("Liverpool have signed 18-year-old defender Giovanni Leoni from Italian side Parma for a fee of £26m plus add-ons. Leoni, who Reds boss Arne Slot confirmed the club had agreed a deal for earlier this week, has signed a contract until 2031. The Italy Under-19s centre-back began his career at Padova and had a short spell at Sampdoria before joining Parma last summer. Leoni played 17 games in his first campaign in Serie A last season, attracting interest from both Milan clubs, but has opted for a move abroad.")

# business
avalia_sentenca("Shares in Intel jumped by more than 7% on Thursday, following reports that the Trump administration is in talks to take a stake in the chipmaker. The reported deal would support the technology firm's plans to build a manufacturing hub in Ohio, according to Bloomberg, which said the size of the potential stake is not clear. White House spokesman Kush Desai said 'discussion about hypothetical deals should be regarded as speculation unless officially announced by the administration'.")

# tech
avalia_sentenca("Apple on Tuesday released its sixth developer beta of its mobile operating system, iOS 26, along with updates for iPad, Apple Watch, Apple TV, Mac, and others. As the new software gets closer to its September public launch date, the changes and tweaks aren't as significant, but there are still some surprises to be found. In beta 6, that includes a new selection of ringtones, a final decision on the Camera swiping direction controversy, a new onboarding experience, and other updates. There was quite a bit of backlash over the way Apple changed how the Camera mode switcher responds to touch. Users complained that the change, introduced in prior betas, broke their muscle memory, since the swipe gesture on this interface would reverse the scroll direction they were used to — which was the same design paradigm that all other native Apple apps use. In beta 5, Apple responded to these complaints by introducing a new Settings toggle that would allow you to switch back to 'Classic Mode', which would once again make the Camera app work like all others.")

# politics
avalia_sentenca("Watching the president of France and the prime minister close up was to see two men under the cosh, behind in the opinion polls and fighting for what they see as the essence of their political creed. It boils down to this - how do Sir Keir Starmer and Emmanuel Macron make the case for what they see as the virtues of patience, nuance, subtleties and trade-offs in an era of growing impatience at the perceived repeated failures of those in high office? There were just two lecterns and two speakers at the news conference the leaders hosted, but two other parties hovered in the air. Reform UK and National Rally, the party of Jordan Bardella and Marine Le Pen, were never mentioned explicitly. Nor were their leaders. But they were repeatedly mentioned implicitly.")


## Atividade 4: Análise dos Resultados

Responda as seguintes perguntas apresentando uma justificativa breve (poucas frases).

1. Considerando o gráfico de evolução da função de perda, responda as seguintes perguntas: 
   1. Existem indícios de *underfitting* (subajuste) do modelo?
   2. Existem indícios de *overfitting* (superajuste) do modelo?

2. Considerando os resultados da análise quantitativa, responda as seguintes perguntas: 
   1. O modelo apresentou uma boa generalização? 
   2. Em quais classes o modelo apresentou melhor e pior performance de acordo com o `f1-score`?

3. Quais conclusões podem ser tiradas com a análise qualitativa? 



---

*Clique duas vezes para adicionar sua resposta...*

---
